In [ ]:
import pandas as pd

In [ ]:
y = pd.read_csv('youtube.csv')
t = pd.read_csv('twitch.csv')

In [ ]:
t.info()

In [ ]:
y.info()

In [ ]:
y['language'].unique()

In [ ]:
import pandas as pd
from typing import Set, List


SENTIMENT_COV: Set[str] = {           
    "Arabic", "English", "French", "German", "Hindi", "Italian", "Spanish", "Portuguese"
}
TOXICITY_COV: Set[str] = {           
    "Arabic", "English", "German", "Spanish", "Russian", "Ukranian",
    "Chinese", "Amharic", "Hindi", 'Japanese', 'Italian', 'French', 
}

def prepare_platform_df(df: pd.DataFrame, platform: str) -> pd.DataFrame:
    """
    Ritorna:
        streamer_name | language | msg_len | hard_toxic_neg |
        unsupported_sent | unsupported_tox | platform
    """
    required: List[str] = [
        "streamer_name","language","message",
        "sentiment_label","sentiment_score",
        "toxicity_label","toxicity_score"
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{platform}: mancano colonne {missing}")

    msg_len = (
        df["message"].fillna("")
          .str.split().str.len().astype("int32")
    )

    unsup_sent = ~df["language"].isin(SENTIMENT_COV)
    unsup_tox  = ~df["language"].isin(TOXICITY_COV)

    tox_thresh  = 0.95 + 0.02 * unsup_tox.astype("int")   # 0.95  o 0.97
    sent_thresh = 0.66 + 0.09 * unsup_sent.astype("int")  # 0.66  o 0.75

    hard_toxic_neg = (
        (df["toxicity_label"]  == "label_1") & (df["toxicity_score"]  >= tox_thresh) &
        (df["sentiment_label"] == "neg")     & (df["sentiment_score"] >= sent_thresh)
    ).astype("int8")

    return pd.DataFrame({
        "streamer_name":    df["streamer_name"],
        "language":         df["language"],
        "msg_len":          msg_len,
        "hard_toxic_neg":   hard_toxic_neg,
        "unsupported_sent": unsup_sent.astype("int8"),
        "unsupported_tox":  unsup_tox.astype("int8"),
        "platform":         platform
    })


In [ ]:
df_t   = prepare_platform_df(t, "twitch")
df_y   = prepare_platform_df(y, "youtube")

final_df = pd.concat([df_t, df_y], ignore_index=True)
print(final_df.head())

In [ ]:
final_df.head()

In [ ]:
final_df.info()

In [ ]:
def missing_values(df):
    missing_values = df.isna().sum()
    
    missing_percentage = (missing_values / len(df)) * 100

    missing_data = pd.DataFrame({
        'Missing Values': missing_values,
        'Percentage': missing_percentage
    })
    
    return missing_data

In [ ]:
missing_values(final_df)

In [ ]:
final_df[final_df.isna().any(axis=1)]

In [ ]:
final_df = final_df.dropna()

In [ ]:
final_df.to_parquet("ready_all_langs.parquet", index=False)